# Imports

In [ ]:
import os
import warnings
import itertools
import datetime as dt
import pandas as pd
import numpy as np
import random # Usado para random.sample e random.shuffle
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from scipy.stats import norm, skew, kurtosis, truncnorm # Importar diretamente as funções
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf
from scipy.optimize import curve_fit
from statsmodels.graphics.gofplots import qqplot
from multiprocessing import Pool
import random
import traceback # Importar para printar o traceback em caso de erro

# Importar nas classes quando usar função standalone

warnings.filterwarnings("ignore")

In [ ]:
!git clone https://github.com/gilbertogilfgp/Mercado-Artificial-Fiis.git
%cd /content/Mercado-Artificial-Fiis

import sys
sys.path.append('/content/Mercado-Artificial-Fiis')

!pip install numpy pandas matplotlib scipy statsmodels yfinance seaborn

from src.simulacao import simular_mercado_e_plotar

Cloning into 'Mercado-Artificial-Fiis'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 72 (delta 29), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 233.74 KiB | 9.35 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/Mercado-Artificial-Fiis


#Funções

## Matriz de referências

In [ ]:
M_cal = [[0.5, 0.9],
         [0.5, 0.9],
         [0.5, 0.9],
         [0.1, 0.3],
         [0.06, 0.2]
         ]
M_cal

[[0.5, 0.9], [0.5, 0.9], [0.5, 0.9], [0.1, 0.3], [0.06, 0.2]]

## **Parâmetros para o MC**

In [ ]:
def parametros_mc(M_cal):
    M_cal_np = np.array(M_cal)
    return np.random.uniform(low=M_cal_np[:, 0], high=M_cal_np[:, 1])

## Rotinas

In [ ]:
def momentos_simulacao(r: np.ndarray)->np.ndarray:
    r = np.asarray(r, float)
    lags = [5, 10, 15, 20]
    autocorrs = acf(r, nlags=max(lags), fft=True)
    autocorr_q = acf(r**2, nlags=2, fft=True)
    acf_r = np.array([autocorrs[lag] for lag in lags])
    return np.array([
        np.mean(r),
        np.var(r, ddof=1),                    # variância amostral
        skew(r, bias=False),                  # skew sem viés
        kurtosis(r, fisher=False, bias=False), # curtose clássica (não-excesso)
        acf_r[0], acf_r[1], acf_r[2], acf_r[3],
        autocorr_q[1], autocorr_q[2]
    ], float)

In [ ]:
def rodar_mercado_R_vezes(parametros_sistema, num_dias, R):
  medias_sim = []
  variancias_sim = []
  assimetrias_sim = []
  curtoses_sim = []
  acf_5 = []
  acf_10 = []
  acf_15 = []
  acf_20 = []
  autocorr_q_1 = []
  autocorr_q_2 = []


  for i in range(R):
    historico_precos_fii, log_returns, volatilidade_rolante, midia, sentimento_medio_ao_longo_dos_dias = \
            simular_mercado_e_plotar(parametros_sistema, num_dias, imprimir=False)

    momentos_sim = momentos_simulacao(log_returns[-num_dias:])
    medias_sim.append(momentos_sim[0])
    variancias_sim.append(momentos_sim[1])
    assimetrias_sim.append(momentos_sim[2])
    curtoses_sim.append(momentos_sim[3])
    acf_5.append(momentos_sim[4])
    acf_10.append(momentos_sim[5])
    acf_15.append(momentos_sim[6])
    acf_20.append(momentos_sim[7])
    autocorr_q_1.append(momentos_sim[8])
    autocorr_q_2.append(momentos_sim[9])

    print(f"R = ", i)

    mom_sim_R = np.array([np.mean(medias_sim), np.mean(variancias_sim), np.mean(assimetrias_sim), np.mean(curtoses_sim), np.mean(acf_5), np.mean(acf_10), np.mean(acf_15), np.mean(acf_20), np.mean(autocorr_q_1), np.mean(autocorr_q_2)])

    #print(f"R = ", i, momentos_sim)

  #print(f"Médias: ", mom_sim_R)


  return mom_sim_R

# Rodar e Salvar

In [ ]:
n_rodadas = 100
R = 10
num_dias = 60

parametros_R = []
medias_sim_R = []
variancias_sim_R = []
assimetrias_sim_R = []
curtoses_sim_R = []
acf_5_sim_R = []
acf_10_sim_R = []
acf_15_sim_R = []
acf_20_sim_R = []
acf_q1_sim_R = []
acf_q2_sim_R = []


for rodada in range(n_rodadas):
    # 1. Gerar um conjunto aleatório de parâmetros
    parametros_sistema = parametros_mc(M_cal)
    parametros_R.append(parametros_sistema)

    # 2. Rodar a simulação múltiplas vezes e extrair momentos médios
    mom_sim_R = rodar_mercado_R_vezes(parametros_sistema, num_dias, R)
    # Espera-se que mom_sim_R seja o vetor retornado por momentos_simulacao()

    # 3. Armazenar cada momento
    medias_sim_R.append(mom_sim_R[0])
    variancias_sim_R.append(mom_sim_R[1])
    assimetrias_sim_R.append(mom_sim_R[2])
    curtoses_sim_R.append(mom_sim_R[3])
    acf_5_sim_R.append(mom_sim_R[4])
    acf_10_sim_R.append(mom_sim_R[5])
    acf_15_sim_R.append(mom_sim_R[6])
    acf_20_sim_R.append(mom_sim_R[7])
    acf_q1_sim_R.append(mom_sim_R[8])
    acf_q2_sim_R.append(mom_sim_R[9])


    print(f"Rodada: {rodada}")

# ==============================================================
# Construir DataFrame com resultados
# ==============================================================
df_resultados = pd.DataFrame(parametros_R, columns=['a0', 'b0', 'c0', 'beta', 'p5'])

df_resultados['Media']       = medias_sim_R
df_resultados['Variancia']   = variancias_sim_R
df_resultados['Assimetria']  = assimetrias_sim_R
df_resultados['Curtose']     = curtoses_sim_R
df_resultados['ACF_5']       = acf_5_sim_R
df_resultados['ACF_10']      = acf_10_sim_R
df_resultados['ACF_15']      = acf_15_sim_R
df_resultados['ACF_20']      = acf_20_sim_R
df_resultados['ACF_Q1']      = acf_q1_sim_R
df_resultados['ACF_Q2']      = acf_q2_sim_R


# ==============================================================
# Salvar e fazer download do DataFrame
# ==============================================================

# Salvar o DataFrame em um arquivo CSV
df_resultados.to_csv('resultados_simulacao_28mar.csv', index=False)

# Criar um link para download no Google Colab
from google.colab import files
files.download('resultados_simulacao_28mar.csv')

A saída de streaming foi truncada nas últimas 5000 linhas.
⚡ Choque aleatório positivo no dia 60 (intensidade 0.67)
R =  5
Total de Agentes: 800
⚡ Choque aleatório negativo no dia 9 (intensidade 0.58)

💥 Choque NEGATIVO aplicado no dia 30
R =  6
Total de Agentes: 800
⚡ Choque aleatório negativo no dia 15 (intensidade 0.58)

💥 Choque NEGATIVO aplicado no dia 30
R =  7
Total de Agentes: 800

💥 Choque NEGATIVO aplicado no dia 30
R =  8
Total de Agentes: 800
⚡ Choque aleatório negativo no dia 11 (intensidade 0.49)

💥 Choque NEGATIVO aplicado no dia 30
⚡ Choque aleatório negativo no dia 45 (intensidade 0.62)
R =  9
Rodada: 10
Total de Agentes: 800
⚡ Choque aleatório negativo no dia 3 (intensidade 0.39)
⚡ Choque aleatório negativo no dia 28 (intensidade 0.32)

💥 Choque NEGATIVO aplicado no dia 30
⚡ Choque aleatório negativo no dia 52 (intensidade 0.64)
R =  0
Total de Agentes: 800
⚡ Choque aleatório negativo no dia 22 (intensidade 0.40)

💥 Choque NEGATIVO aplicado no dia 30
⚡ Choque aleatóri

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>